# Understanding Dataset

### To identify online payment fraud with machine learning, we need to train a machine learning model for classifying fraudulent and non-fraudulent payments. For this, we need a dataset containing information about online payment fraud, so that we can understand what type of transactions lead to fraud.

### Below are all the columns from the dataset we are using here:

#### step: represents a unit of time where 1 step equals 1 hour
#### type: type of online transaction
#### amount: the amount of the transaction
#### nameOrig: customer starting the transaction
#### oldbalanceOrg: balance before the transaction
#### newbalanceOrig: balance after the transaction
#### nameDest: recipient of the transaction
#### oldbalanceDest: initial balance of recipient before the transaction
#### newbalanceDest: the new balance of recipient after the transaction
#### isFraud: fraud transaction

# ---------------------------------------------------------------------------------------

# Importing Liberaries & Dataset

In [2]:
import pandas as pd
import numpy as np
import csv 
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score

In [3]:

"""
with open("score.csv","+a") as file:
    writer = csv.writer(file)
    writer.writerow(["Module","Accuracy Score","Precision Score","Recall Score","F1 Score"])
"""

'\nwith open("score.csv","+a") as file:\n    writer = csv.writer(file)\n    writer.writerow(["Module","Accuracy Score","Precision Score","Recall Score","F1 Score"])\n'

In [4]:
data = pd.read_csv(r"C:\Users\ASUS\Desktop\Project\CC\onlinefraud.csv")
print(data.head())

   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  


In [5]:
data.shape

(6362620, 11)

In [6]:
data.dtypes

step                int64
type               object
amount            float64
nameOrig           object
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest           object
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object

In [7]:
# Checking Null Values

print(data.isnull().sum())

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64


In [8]:
print(data.type.value_counts())

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64


In [9]:
type = data["type"].value_counts()
transactions = type.index
quantity = type.values
figure = px.pie(data, values=quantity,hole=0.5,title="Distribution of Transaction Type", names= type)
figure.show()

In [10]:
# Now let’s transform the categorical features into numerical. Here we will also transform the values of the isFraud column into 
# No Fraud and Fraud labels to have a better understanding of the output
# Changing CASH_OUT to 1, PAYMENT to 2, CASH_IN to 3, TRANSFER to 4 and DEBIT to 5 

data["type"] = data["type"].map({"CASH_OUT": 1, "PAYMENT": 2, "CASH_IN": 3, "TRANSFER": 4, "DEBIT": 5})
#data["isFraud"] = data["isFraud"].map({0: "No Fraud", 1: "Fraud"})
print(data.head())

   step  type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1     2   9839.64  C1231006815       170136.0       160296.36   
1     1     2   1864.28  C1666544295        21249.0        19384.72   
2     1     4    181.00  C1305486145          181.0            0.00   
3     1     1    181.00   C840083671          181.0            0.00   
4     1     2  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  


In [11]:
# splitting the data

X = np.array(data[["type", "amount", "oldbalanceOrg", "newbalanceOrig"]])
y = np.array(data[["isFraud"]])

In [12]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Logistic regression

In [29]:
model = LogisticRegression()

fold_accuracies = []
fold_f1 = []
fold_precision = []
fold_recall = []
fold_roc_auc = []

for fold, (train_index, val_index) in enumerate(skf.split(X, y), 1):
    X_train, X_test = X[train_index], X[val_index]
    y_train, y_test = y[train_index], y[val_index]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    fold_accuracies.append(accuracy)

    precision = precision_score(y_test, y_pred)
    fold_precision.append(precision)

    f1 = f1_score(y_test, y_pred)
    fold_f1.append(f1)

    recall = recall_score(y_test, y_pred)
    fold_recall.append(recall)

    auc = roc_auc_score(y_test, y_pred)
    fold_roc_auc.append(auc)

    print(f"Fold {fold} Accuracy Score: {accuracy:.4f}")
    print(f"Fold {fold} Precision Score: {precision:.4f}")
    print(f"Fold {fold} Recall Score: {recall:.4f}")
    print(f"Fold {fold} f1 Score: {f1:.4f}")
    print(f"Fold {fold} ROC-AUC Score: {auc:.4f}")
    print("\n")


Fold 1 Accuracy Score: 0.9991
Fold 1 Precision Score: 0.8896
Fold 1 Recall Score: 0.3532
Fold 1 f1 Score: 0.5057
Fold 1 ROC-AUC Score: 0.6766


Fold 2 Accuracy Score: 0.9991
Fold 2 Precision Score: 0.7892
Fold 2 Recall Score: 0.4470
Fold 2 f1 Score: 0.5708
Fold 2 ROC-AUC Score: 0.7234


Fold 3 Accuracy Score: 0.9991
Fold 3 Precision Score: 0.8665
Fold 3 Recall Score: 0.3873
Fold 3 f1 Score: 0.5354
Fold 3 ROC-AUC Score: 0.6936


Fold 4 Accuracy Score: 0.9991
Fold 4 Precision Score: 0.7605
Fold 4 Recall Score: 0.4872
Fold 4 f1 Score: 0.5939
Fold 4 ROC-AUC Score: 0.7435


Fold 5 Accuracy Score: 0.9991
Fold 5 Precision Score: 0.8825
Fold 5 Recall Score: 0.3569
Fold 5 f1 Score: 0.5082
Fold 5 ROC-AUC Score: 0.6784


Fold 6 Accuracy Score: 0.9991
Fold 6 Precision Score: 0.9570
Fold 6 Recall Score: 0.3252
Fold 6 f1 Score: 0.4855
Fold 6 ROC-AUC Score: 0.6626


Fold 7 Accuracy Score: 0.9991
Fold 7 Precision Score: 0.8631
Fold 7 Recall Score: 0.3532
Fold 7 f1 Score: 0.5013
Fold 7 ROC-AUC Score: 0

In [34]:
print(f"Mean Accuracy Score: {np.mean(fold_accuracies):.4f} with Standard Deviation: {np.std(fold_accuracies):.4f}")
print(f"Mean Precision Score: {np.mean(fold_precision):.4f} with Standard Deviation: {np.std(fold_precision):.4f}")
print(f"Mean Recall Score: {np.mean(fold_recall):.4f} with Standard Deviation: {np.std(fold_recall):.4f}")
print(f"Mean F1 Score: {np.mean(fold_f1):.4f} with Standard Deviation: {np.std(fold_f1):.4f}")
print(f"Mean AUC-ROC Score: {np.mean(fold_roc_auc):.4f} with Standard Deviation: {np.std(fold_roc_auc):.4f}")

Mean Accuracy Score: 0.9991 with Standard Deviation: 0.0000
Mean Precision Score: 0.8653 with Standard Deviation: 0.0520
Mean Recall Score: 0.3840 with Standard Deviation: 0.0458
Mean F1 Score: 0.5284 with Standard Deviation: 0.0313
Mean AUC-ROC Score: 0.6920 with Standard Deviation: 0.0229


# Decision Tree Classifier

In [35]:
model = DecisionTreeClassifier()

fold_accuracies = []
fold_f1 = []
fold_precision = []
fold_recall = []
fold_roc_auc = []

for fold, (train_index, val_index) in enumerate(skf.split(X, y), 1):
    X_train, X_test = X[train_index], X[val_index]
    y_train, y_test = y[train_index], y[val_index]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    fold_accuracies.append(accuracy)

    precision = precision_score(y_test, y_pred)
    fold_precision.append(precision)

    f1 = f1_score(y_test, y_pred)
    fold_f1.append(f1)

    recall = recall_score(y_test, y_pred)
    fold_recall.append(recall)

    auc = roc_auc_score(y_test, y_pred)
    fold_roc_auc.append(auc)

    print(f"Fold {fold} Accuracy Score: {accuracy:.4f}")
    print(f"Fold {fold} Precision Score: {precision:.4f}")
    print(f"Fold {fold} Recall Score: {recall:.4f}")
    print(f"Fold {fold} f1 Score: {f1:.4f}")
    print(f"Fold {fold} ROC-AUC Score: {auc:.4f}")
    print("\n")


Fold 1 Accuracy Score: 0.9997
Fold 1 Precision Score: 0.8874
Fold 1 Recall Score: 0.8831
Fold 1 f1 Score: 0.8852
Fold 1 ROC-AUC Score: 0.9415


Fold 2 Accuracy Score: 0.9997
Fold 2 Precision Score: 0.8933
Fold 2 Recall Score: 0.8563
Fold 2 f1 Score: 0.8744
Fold 2 ROC-AUC Score: 0.9281


Fold 3 Accuracy Score: 0.9997
Fold 3 Precision Score: 0.8953
Fold 3 Recall Score: 0.8855
Fold 3 f1 Score: 0.8904
Fold 3 ROC-AUC Score: 0.9427


Fold 4 Accuracy Score: 0.9997
Fold 4 Precision Score: 0.8898
Fold 4 Recall Score: 0.8952
Fold 4 f1 Score: 0.8925
Fold 4 ROC-AUC Score: 0.9476


Fold 5 Accuracy Score: 0.9998
Fold 5 Precision Score: 0.9019
Fold 5 Recall Score: 0.9074
Fold 5 f1 Score: 0.9047
Fold 5 ROC-AUC Score: 0.9537


Fold 6 Accuracy Score: 0.9997
Fold 6 Precision Score: 0.9017
Fold 6 Recall Score: 0.8831
Fold 6 f1 Score: 0.8923
Fold 6 ROC-AUC Score: 0.9415


Fold 7 Accuracy Score: 0.9997
Fold 7 Precision Score: 0.8915
Fold 7 Recall Score: 0.8709
Fold 7 f1 Score: 0.8811
Fold 7 ROC-AUC Score: 0

In [36]:
print(f"Mean Accuracy Score: {np.mean(fold_accuracies):.4f} with Standard Deviation: {np.std(fold_accuracies):.4f}")
print(f"Mean Precision Score: {np.mean(fold_precision):.4f} with Standard Deviation: {np.std(fold_precision):.4f}")
print(f"Mean Recall Score: {np.mean(fold_recall):.4f} with Standard Deviation: {np.std(fold_recall):.4f}")
print(f"Mean F1 Score: {np.mean(fold_f1):.4f} with Standard Deviation: {np.std(fold_f1):.4f}")
print(f"Mean AUC-ROC Score: {np.mean(fold_roc_auc):.4f} with Standard Deviation: {np.std(fold_roc_auc):.4f}")

Mean Accuracy Score: 0.9997 with Standard Deviation: 0.0000
Mean Precision Score: 0.8959 with Standard Deviation: 0.0057
Mean Recall Score: 0.8835 with Standard Deviation: 0.0136
Mean F1 Score: 0.8896 with Standard Deviation: 0.0085
Mean AUC-ROC Score: 0.9417 with Standard Deviation: 0.0068


# Random Forest

In [13]:
model = RandomForestClassifier()

fold_accuracies = []
fold_f1 = []
fold_precision = []
fold_recall = []
fold_roc_auc = []

for fold, (train_index, val_index) in enumerate(skf.split(X, y), 1):
    X_train, X_test = X[train_index], X[val_index]
    y_train, y_test = y[train_index], y[val_index]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    fold_accuracies.append(accuracy)

    precision = precision_score(y_test, y_pred)
    fold_precision.append(precision)

    f1 = f1_score(y_test, y_pred)
    fold_f1.append(f1)

    recall = recall_score(y_test, y_pred)
    fold_recall.append(recall)

    auc = roc_auc_score(y_test, y_pred)
    fold_roc_auc.append(auc)

    print(f"Fold {fold} Accuracy Score: {accuracy:.4f}")
    print(f"Fold {fold} Precision Score: {precision:.4f}")
    print(f"Fold {fold} Recall Score: {recall:.4f}")
    print(f"Fold {fold} f1 Score: {f1:.4f}")
    print(f"Fold {fold} ROC-AUC Score: {auc:.4f}")
    print("\n")


Fold 1 Accuracy Score: 0.9997
Fold 1 Precision Score: 0.9025
Fold 1 Recall Score: 0.8904
Fold 1 f1 Score: 0.8964
Fold 1 ROC-AUC Score: 0.9451


Fold 2 Accuracy Score: 0.9997
Fold 2 Precision Score: 0.9082
Fold 2 Recall Score: 0.8794
Fold 2 f1 Score: 0.8936
Fold 2 ROC-AUC Score: 0.9397


Fold 3 Accuracy Score: 0.9997
Fold 3 Precision Score: 0.9055
Fold 3 Recall Score: 0.8989
Fold 3 f1 Score: 0.9022
Fold 3 ROC-AUC Score: 0.9494


Fold 4 Accuracy Score: 0.9997
Fold 4 Precision Score: 0.9026
Fold 4 Recall Score: 0.8916
Fold 4 f1 Score: 0.8971
Fold 4 ROC-AUC Score: 0.9457


Fold 5 Accuracy Score: 0.9998
Fold 5 Precision Score: 0.9083
Fold 5 Recall Score: 0.9050
Fold 5 f1 Score: 0.9067
Fold 5 ROC-AUC Score: 0.9524


Fold 6 Accuracy Score: 0.9998
Fold 6 Precision Score: 0.9165
Fold 6 Recall Score: 0.8952
Fold 6 f1 Score: 0.9057
Fold 6 ROC-AUC Score: 0.9476


Fold 7 Accuracy Score: 0.9997
Fold 7 Precision Score: 0.9015
Fold 7 Recall Score: 0.8806
Fold 7 f1 Score: 0.8909
Fold 7 ROC-AUC Score: 0

In [14]:
print(f"Mean Accuracy Score: {np.mean(fold_accuracies):.4f} with Standard Deviation: {np.std(fold_accuracies):.4f}")
print(f"Mean Precision Score: {np.mean(fold_precision):.4f} with Standard Deviation: {np.std(fold_precision):.4f}")
print(f"Mean Recall Score: {np.mean(fold_recall):.4f} with Standard Deviation: {np.std(fold_recall):.4f}")
print(f"Mean F1 Score: {np.mean(fold_f1):.4f} with Standard Deviation: {np.std(fold_f1):.4f}")
print(f"Mean AUC-ROC Score: {np.mean(fold_roc_auc):.4f} with Standard Deviation: {np.std(fold_roc_auc):.4f}")

Mean Accuracy Score: 0.9997 with Standard Deviation: 0.0000
Mean Precision Score: 0.9090 with Standard Deviation: 0.0082
Mean Recall Score: 0.8925 with Standard Deviation: 0.0089
Mean F1 Score: 0.9007 with Standard Deviation: 0.0074
Mean AUC-ROC Score: 0.9462 with Standard Deviation: 0.0044


# Gaussian Naive Bayes

In [15]:
model = GaussianNB()

fold_accuracies = []
fold_f1 = []
fold_precision = []
fold_recall = []
fold_roc_auc = []

for fold, (train_index, val_index) in enumerate(skf.split(X, y), 1):
    X_train, X_test = X[train_index], X[val_index]
    y_train, y_test = y[train_index], y[val_index]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    fold_accuracies.append(accuracy)

    precision = precision_score(y_test, y_pred)
    fold_precision.append(precision)

    f1 = f1_score(y_test, y_pred)
    fold_f1.append(f1)

    recall = recall_score(y_test, y_pred)
    fold_recall.append(recall)

    auc = roc_auc_score(y_test, y_pred)
    fold_roc_auc.append(auc)

    print(f"Fold {fold} Accuracy Score: {accuracy:.4f}")
    print(f"Fold {fold} Precision Score: {precision:.4f}")
    print(f"Fold {fold} Recall Score: {recall:.4f}")
    print(f"Fold {fold} f1 Score: {f1:.4f}")
    print(f"Fold {fold} ROC-AUC Score: {auc:.4f}")
    print("\n")


Fold 1 Accuracy Score: 0.9946
Fold 1 Precision Score: 0.0426
Fold 1 Recall Score: 0.1474
Fold 1 f1 Score: 0.0661
Fold 1 ROC-AUC Score: 0.5716


Fold 2 Accuracy Score: 0.9946
Fold 2 Precision Score: 0.0501
Fold 2 Recall Score: 0.1778
Fold 2 f1 Score: 0.0782
Fold 2 ROC-AUC Score: 0.5867


Fold 3 Accuracy Score: 0.9945
Fold 3 Precision Score: 0.0445
Fold 3 Recall Score: 0.1596
Fold 3 f1 Score: 0.0696
Fold 3 ROC-AUC Score: 0.5776


Fold 4 Accuracy Score: 0.9945
Fold 4 Precision Score: 0.0492
Fold 4 Recall Score: 0.1778
Fold 4 f1 Score: 0.0771
Fold 4 ROC-AUC Score: 0.5867


Fold 5 Accuracy Score: 0.9944
Fold 5 Precision Score: 0.0472
Fold 5 Recall Score: 0.1730
Fold 5 f1 Score: 0.0742
Fold 5 ROC-AUC Score: 0.5842


Fold 6 Accuracy Score: 0.9944
Fold 6 Precision Score: 0.0431
Fold 6 Recall Score: 0.1559
Fold 6 f1 Score: 0.0676
Fold 6 ROC-AUC Score: 0.5757


Fold 7 Accuracy Score: 0.9946
Fold 7 Precision Score: 0.0492
Fold 7 Recall Score: 0.1754
Fold 7 f1 Score: 0.0768
Fold 7 ROC-AUC Score: 0

In [16]:
print(f"Mean Accuracy Score: {np.mean(fold_accuracies):.4f} with Standard Deviation: {np.std(fold_accuracies):.4f}")
print(f"Mean Precision Score: {np.mean(fold_precision):.4f} with Standard Deviation: {np.std(fold_precision):.4f}")
print(f"Mean Recall Score: {np.mean(fold_recall):.4f} with Standard Deviation: {np.std(fold_recall):.4f}")
print(f"Mean F1 Score: {np.mean(fold_f1):.4f} with Standard Deviation: {np.std(fold_f1):.4f}")
print(f"Mean AUC-ROC Score: {np.mean(fold_roc_auc):.4f} with Standard Deviation: {np.std(fold_roc_auc):.4f}")

Mean Accuracy Score: 0.9945 with Standard Deviation: 0.0001
Mean Precision Score: 0.0467 with Standard Deviation: 0.0028
Mean Recall Score: 0.1669 with Standard Deviation: 0.0105
Mean F1 Score: 0.0730 with Standard Deviation: 0.0044
Mean AUC-ROC Score: 0.5813 with Standard Deviation: 0.0052


# Ada Boost

In [17]:
from sklearn.ensemble import AdaBoostClassifier
model = AdaBoostClassifier()

fold_accuracies = []
fold_f1 = []
fold_precision = []
fold_recall = []
fold_roc_auc = []

for fold, (train_index, val_index) in enumerate(skf.split(X, y), 1):
    X_train, X_test = X[train_index], X[val_index]
    y_train, y_test = y[train_index], y[val_index]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    fold_accuracies.append(accuracy)

    precision = precision_score(y_test, y_pred)
    fold_precision.append(precision)

    f1 = f1_score(y_test, y_pred)
    fold_f1.append(f1)

    recall = recall_score(y_test, y_pred)
    fold_recall.append(recall)

    auc = roc_auc_score(y_test, y_pred)
    fold_roc_auc.append(auc)

    print(f"Fold {fold} Accuracy Score: {accuracy:.4f}")
    print(f"Fold {fold} Precision Score: {precision:.4f}")
    print(f"Fold {fold} Recall Score: {recall:.4f}")
    print(f"Fold {fold} f1 Score: {f1:.4f}")
    print(f"Fold {fold} ROC-AUC Score: {auc:.4f}")
    print("\n")


Fold 1 Accuracy Score: 0.9989
Fold 1 Precision Score: 0.8119
Fold 1 Recall Score: 0.2156
Fold 1 f1 Score: 0.3407
Fold 1 ROC-AUC Score: 0.6078


Fold 2 Accuracy Score: 0.9988
Fold 2 Precision Score: 0.7921
Fold 2 Recall Score: 0.0974
Fold 2 f1 Score: 0.1735
Fold 2 ROC-AUC Score: 0.5487


Fold 3 Accuracy Score: 0.9989
Fold 3 Precision Score: 0.7971
Fold 3 Recall Score: 0.2010
Fold 3 f1 Score: 0.3210
Fold 3 ROC-AUC Score: 0.6005


Fold 4 Accuracy Score: 0.9988
Fold 4 Precision Score: 0.7757
Fold 4 Recall Score: 0.1011
Fold 4 f1 Score: 0.1789
Fold 4 ROC-AUC Score: 0.5505


Fold 5 Accuracy Score: 0.9988
Fold 5 Precision Score: 0.7921
Fold 5 Recall Score: 0.0974
Fold 5 f1 Score: 0.1735
Fold 5 ROC-AUC Score: 0.5487


Fold 6 Accuracy Score: 0.9988
Fold 6 Precision Score: 0.7451
Fold 6 Recall Score: 0.0926
Fold 6 f1 Score: 0.1647
Fold 6 ROC-AUC Score: 0.5463


Fold 7 Accuracy Score: 0.9988
Fold 7 Precision Score: 0.8155
Fold 7 Recall Score: 0.1023
Fold 7 f1 Score: 0.1818
Fold 7 ROC-AUC Score: 0

In [18]:
print(f"Mean Accuracy Score: {np.mean(fold_accuracies):.4f} with Standard Deviation: {np.std(fold_accuracies):.4f}")
print(f"Mean Precision Score: {np.mean(fold_precision):.4f} with Standard Deviation: {np.std(fold_precision):.4f}")
print(f"Mean Recall Score: {np.mean(fold_recall):.4f} with Standard Deviation: {np.std(fold_recall):.4f}")
print(f"Mean F1 Score: {np.mean(fold_f1):.4f} with Standard Deviation: {np.std(fold_f1):.4f}")
print(f"Mean AUC-ROC Score: {np.mean(fold_roc_auc):.4f} with Standard Deviation: {np.std(fold_roc_auc):.4f}")

Mean Accuracy Score: 0.9988 with Standard Deviation: 0.0000
Mean Precision Score: 0.7860 with Standard Deviation: 0.0351
Mean Recall Score: 0.1314 with Standard Deviation: 0.0463
Mean F1 Score: 0.2221 with Standard Deviation: 0.0664
Mean AUC-ROC Score: 0.5657 with Standard Deviation: 0.0232


# Gradient Boosting

In [24]:
from sklearn.ensemble import GradientBoostingRegressor
model = GradientBoostingRegressor()

fold_accuracies = []
fold_f1 = []
fold_precision = []
fold_recall = []
fold_roc_auc = []

for fold, (train_index, val_index) in enumerate(skf.split(X, y), 1):
    X_train, X_test = X[train_index], X[val_index]
    y_train, y_test = y[train_index], y[val_index]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    '''
    accuracy = accuracy_score(y_test, y_pred)
    fold_accuracies.append(accuracy)

    precision = precision_score(y_test, y_pred)
    fold_precision.append(precision)

    f1 = f1_score(y_test, y_pred)
    fold_f1.append(f1)

    recall = recall_score(y_test, y_pred)
    fold_recall.append(recall)
    '''
    auc = roc_auc_score(y_test, y_pred)
    fold_roc_auc.append(auc)

    #print(f"Fold {fold} Accuracy Score: {accuracy:.4f}")
    #print(f"Fold {fold} Precision Score: {precision:.4f}")
    #print(f"Fold {fold} Recall Score: {recall:.4f}")
    #print(f"Fold {fold} f1 Score: {f1:.4f}")
    print(f"Fold {fold} ROC-AUC Score: {auc:.4f}")
    print("\n")


Fold 1 ROC-AUC Score: 0.9905


Fold 2 ROC-AUC Score: 0.9888


Fold 3 ROC-AUC Score: 0.9862


Fold 4 ROC-AUC Score: 0.9888


Fold 5 ROC-AUC Score: 0.9901


Fold 6 ROC-AUC Score: 0.9883


Fold 7 ROC-AUC Score: 0.9876


Fold 8 ROC-AUC Score: 0.9863


Fold 9 ROC-AUC Score: 0.9896


Fold 10 ROC-AUC Score: 0.9883




In [25]:
"""
print(f"Mean Accuracy Score: {np.mean(fold_accuracies):.4f} with Standard Deviation: {np.std(fold_accuracies):.4f}")
print(f"Mean Precision Score: {np.mean(fold_precision):.4f} with Standard Deviation: {np.std(fold_precision):.4f}")
print(f"Mean Recall Score: {np.mean(fold_recall):.4f} with Standard Deviation: {np.std(fold_recall):.4f}")
print(f"Mean F1 Score: {np.mean(fold_f1):.4f} with Standard Deviation: {np.std(fold_f1):.4f}")
"""
print(f"Mean AUC-ROC Score: {np.mean(fold_roc_auc):.4f} with Standard Deviation: {np.std(fold_roc_auc):.4f}")

Mean AUC-ROC Score: 0.9885 with Standard Deviation: 0.0014


# KNN

In [26]:
model = KNeighborsClassifier(n_neighbors = 7)

fold_accuracies = []
fold_f1 = []
fold_precision = []
fold_recall = []
fold_roc_auc = []

for fold, (train_index, val_index) in enumerate(skf.split(X, y), 1):
    X_train, X_test = X[train_index], X[val_index]
    y_train, y_test = y[train_index], y[val_index]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    fold_accuracies.append(accuracy)

    precision = precision_score(y_test, y_pred)
    fold_precision.append(precision)

    f1 = f1_score(y_test, y_pred)
    fold_f1.append(f1)

    recall = recall_score(y_test, y_pred)
    fold_recall.append(recall)

    auc = roc_auc_score(y_test, y_pred)
    fold_roc_auc.append(auc)

    print(f"Fold {fold} Accuracy Score: {accuracy:.4f}")
    print(f"Fold {fold} Precision Score: {precision:.4f}")
    print(f"Fold {fold} Recall Score: {recall:.4f}")
    print(f"Fold {fold} f1 Score: {f1:.4f}")
    print(f"Fold {fold} ROC-AUC Score: {auc:.4f}")
    print("\n")


Fold 1 Accuracy Score: 0.9996
Fold 1 Precision Score: 0.7955
Fold 1 Recall Score: 0.8952
Fold 1 f1 Score: 0.8424
Fold 1 ROC-AUC Score: 0.9475


Fold 2 Accuracy Score: 0.9996
Fold 2 Precision Score: 0.8046
Fold 2 Recall Score: 0.8928
Fold 2 f1 Score: 0.8464
Fold 2 ROC-AUC Score: 0.9463


Fold 3 Accuracy Score: 0.9996
Fold 3 Precision Score: 0.8026
Fold 3 Recall Score: 0.8867
Fold 3 f1 Score: 0.8426
Fold 3 ROC-AUC Score: 0.9432


Fold 4 Accuracy Score: 0.9995
Fold 4 Precision Score: 0.7862
Fold 4 Recall Score: 0.8867
Fold 4 f1 Score: 0.8334
Fold 4 ROC-AUC Score: 0.9432


Fold 5 Accuracy Score: 0.9996
Fold 5 Precision Score: 0.7873
Fold 5 Recall Score: 0.8928
Fold 5 f1 Score: 0.8368
Fold 5 ROC-AUC Score: 0.9463


Fold 6 Accuracy Score: 0.9996
Fold 6 Precision Score: 0.7956
Fold 6 Recall Score: 0.8770
Fold 6 f1 Score: 0.8343
Fold 6 ROC-AUC Score: 0.9383


Fold 7 Accuracy Score: 0.9995
Fold 7 Precision Score: 0.7859
Fold 7 Recall Score: 0.8855
Fold 7 f1 Score: 0.8328
Fold 7 ROC-AUC Score: 0

In [27]:
print(f"Mean Accuracy Score: {np.mean(fold_accuracies):.4f} with Standard Deviation: {np.std(fold_accuracies):.4f}")
print(f"Mean Precision Score: {np.mean(fold_precision):.4f} with Standard Deviation: {np.std(fold_precision):.4f}")
print(f"Mean Recall Score: {np.mean(fold_recall):.4f} with Standard Deviation: {np.std(fold_recall):.4f}")
print(f"Mean F1 Score: {np.mean(fold_f1):.4f} with Standard Deviation: {np.std(fold_f1):.4f}")
print(f"Mean AUC-ROC Score: {np.mean(fold_roc_auc):.4f} with Standard Deviation: {np.std(fold_roc_auc):.4f}")

Mean Accuracy Score: 0.9996 with Standard Deviation: 0.0000
Mean Precision Score: 0.7991 with Standard Deviation: 0.0108
Mean Recall Score: 0.8875 with Standard Deviation: 0.0103
Mean F1 Score: 0.8409 with Standard Deviation: 0.0084
Mean AUC-ROC Score: 0.9436 with Standard Deviation: 0.0051
